In [ ]:
import pandas as pd

print("Cargando el archivo histórico...")
ruta_excel = 'data/registro-administrativo-historico_2009-2024-inicio.xlsx'
df = pd.read_excel(ruta_excel, sheet_name=0, engine='openpyxl')

# 1. Creamos un diccionario (RENAME) para mapear los nombres originales del Excel a un formato estándar
RENAME = {
    'Periodo': 'anio_lectivo',
    'Codigo_Institucion': 'cod_amie',
    'Nombre_Institucion': 'nombre_institucion',
    'Tipo_Educacion': 'nivel_educacion', 
    'Sostenimiento': 'sostenimiento',
    'Area': 'area',
    'Provincia': 'provincia',
    'Canton': 'canton',
    'Parroquia': 'parroquia',
    'Zona': 'zona',
    'Docentes_Femenino': 'docentes_f',
    'Docentes_Masculino': 'docentes_m',
    'Total_Docentes': 'total_docentes',
    'Estudiantes_Femenino': 'estudiantes_f',
    'Estudiantes_Masculino': 'estudiantes_m',
    'Total_Estudiantes': 'total_estudiantes'
}

# Aplicamos la función rename para reemplazar los nombres en el DataFrame usando el diccionario
df = df.rename(columns=RENAME)
print("\n1. Columnas renombradas exitosamente.")

# Filtramos el DataFrame para conservar únicamente las 16 columnas declaradas en el diccionario RENAME, 
# descartando el resto de variables
columnas_finales = list(RENAME.values())
df = df[columnas_finales]

# 2. Identificamos las métricas y reemplazamos los valores nulos por 0. forzamos el tipo de dato a entero (int) para evitar decimales incongruentes en conteos de personas.
nums_limpios = ['total_docentes', 'total_estudiantes', 'estudiantes_f', 'estudiantes_m']
df[nums_limpios] = df[nums_limpios].fillna(0).astype(int)
print("2. Nulos rellenados con 0.")

# 3. Utilizamos una clave compuesta (Código AMIE + Año Lectivo) para identificar y purgar 
# registros repetidos
filas_antes = df.shape[0]
df = df.drop_duplicates(subset=['cod_amie', 'anio_lectivo'])
filas_despues = df.shape[0]
print(f"3. Duplicados eliminados: {filas_antes - filas_despues} registros purgados.")

# 4. Comprobamos lógicamente que la suma de estudiantes masculinos y femeninos coincida con el total reportado.
df['suma_real'] = df['estudiantes_f'] + df['estudiantes_m']
inconsistentes = df[df['suma_real'] != df['total_estudiantes']]
print(f"4. Consistencia: Se encontraron {inconsistentes.shape[0]} registros inconsistentes.")

# Resolvemos la integridad sobrescribiendo la columna 'total_estudiantes' con la suma real validada
df['total_estudiantes'] = df['suma_real']
df = df.drop(columns=['suma_real']) 
print("   -> Decisión aplicada: 'total_estudiantes' recalculado sumando F + M.")

print("\nLimpieza finalizada")
print("Dimensiones finales:", df.shape)

Cargando el archivo histórico...

1. Columnas renombradas exitosamente.
2. Nulos rellenados con 0 en columnas numéricas clave.
3. Duplicados eliminados: 2 registros purgados.
4. Consistencia: Se encontraron 0 registros inconsistentes.
   -> Decisión aplicada: 'total_estudiantes' recalculado sumando F + M.

¡Limpieza finalizada! Tu dataset está listo para SQLite.
Dimensiones finales: (290308, 16)


In [ ]:
# Importamos create_engine de SQLAlchemy para actuar como puente hacia la base de datos SQLite.
from sqlalchemy import create_engine
import pandas as pd

print("Iniciando...")

# 1. Generamos la conexión local apuntando a un archivo SQLite llamado 'amie_mineduc.db'. 
# Si el archivo no existe, el motor lo creará automáticamente en la carpeta raíz.
engine = create_engine('sqlite:///amie_mineduc.db')

print("Exportando filas a SQLite...")

# 2.    Exportamos el DataFrame procesado hacia una tabla SQL denominada 'instituciones'.
# 'if_exists=replace': Asegura que la tabla se sobrescriba limpiamente si ejecutamos el script de nuevo.
# 'index=False': Evita exportar el índice numérico automático de Pandas como una columna inútil.
df.to_sql('instituciones', engine, if_exists='replace', index=False)

print("¡Éxito! Tu base de datos 'amie_mineduc.db' ha sido creada y poblada.\n")

Encendiendo el motor de base de datos...
Exportando 290,308 filas a SQLite (esto tomará unos segunditos)...
¡Éxito! Tu base de datos 'amie_mineduc.db' ha sido creada y poblada.


In [ ]:
import pandas as pd

# P1: ¿Cómo se distribuye la matrícula total por provincia en el año lectivo más reciente? 
q1 = """
SELECT provincia, SUM(total_estudiantes) as matricula_total
FROM instituciones
WHERE anio_lectivo LIKE '%2022-2023%'
GROUP BY provincia
ORDER BY matricula_total DESC
LIMIT 5;
"""
print("--- P1: Top 5 Provincias por Matrícula (Último año disponible) ---")
print(pd.read_sql(q1, engine))

# P2: ¿Cuál es la proporción de instituciones fiscales vs. particulares por área (urbana/rural) en Loja?
q2 = """
SELECT area, sostenimiento, COUNT(cod_amie) as cantidad_escuelas
FROM instituciones
WHERE provincia = 'LOJA' 
  AND sostenimiento IN ('Fiscal', 'Particular')
  AND anio_lectivo LIKE '%2022-2023%'
GROUP BY area, sostenimiento;
"""
print("\n--- P2: Escuelas en Loja (Área y Sostenimiento) ---")
print(pd.read_sql(q2, engine))

# P3: ¿Cómo evolucionó el número de instituciones activas en Ecuador entre 2015 y 2024? 
q3 = """
SELECT anio_lectivo, COUNT(cod_amie) as total_instituciones
FROM instituciones
WHERE anio_lectivo >= '2015'
GROUP BY anio_lectivo
ORDER BY anio_lectivo;
"""
print("\n--- P3: Evolución de Instituciones Activas (2015-adelante) ---")
print(pd.read_sql(q3, engine))

--- P1: Top 5 Provincias por Matrícula (Último año disponible) ---
   provincia  matricula_total
0     GUAYAS          1094389
1  PICHINCHA           725310
2     MANABI           406559
3   LOS RIOS           233693
4      AZUAY           200018

--- P2: Escuelas en Loja (Área y Sostenimiento) ---
     area sostenimiento  cantidad_escuelas
0   Rural        Fiscal                647
1   Rural    Particular                  1
2  Urbana        Fiscal                316
3  Urbana    Particular                 58

--- P3: Evolución de Instituciones Activas (2015-adelante) ---
       anio_lectivo  total_instituciones
0  2015-2016 Inicio                18624
1  2016-2017 Inicio                17213
2  2017-2018 Inicio                16624
3  2018-2019 Inicio                16555
4  2019-2020 Inicio                16422
5  2020-2021 Inicio                16209
6  2021-2022 Inicio                16095
7  2022-2023 Inicio                15997
